In [ ]:
import numpy as np

def geometric_random_walk(x, mu, sigma, dt=1):
    return x * np.exp(mu * dt + sigma * np.sqrt(dt) * np.random.normal())


In [ ]:
from filterpy.kalman import UnscentedKalmanFilter as UKF
from filterpy.kalman import MerweScaledSigmaPoints

def fx(x, dt):
    """ State transition function for the geometric random walk """
    mu = 0.1  # Example mean
    sigma = 0.01  # Example volatility
    return np.array([geometric_random_walk(x[0], mu, sigma, dt)])

def hx(x):
    """ Measurement function """
    return x

# Initial state
x0 = np.array([1.0])

# Create sigma points
points = MerweScaledSigmaPoints(n=1, alpha=0.1, beta=2.0, kappa=0.0)

# Create UKF instance
ukf = UKF(dim_x=1, dim_z=1, fx=fx, hx=hx, dt=1, points=points)

# Initial state estimate
ukf.x = x0

# State covariance
ukf.P *= 0.1

# Process noise covariance
ukf.Q *= 0.01

# Measurement noise covariance
ukf.R *= 0.1


In [ ]:
import matplotlib.pyplot as plt

# Generate synthetic data
n_steps = 50
mu = 0.1
sigma = 0.9
true_states = [x0[0]]
measurements = [x0[0] + np.random.normal(0, 0.1)]

for _ in range(1, n_steps):
    true_state = geometric_random_walk(true_states[-1], mu, sigma)
    measurement = true_state + np.random.normal(0, 0.1)
    true_states.append(true_state)
    measurements.append(measurement)

# Apply UKF
filtered_states = []

for measurement in measurements:
    ukf.predict()
    ukf.update(np.array([measurement]))
    filtered_states.append(ukf.x[0])

# Plot results
plt.plot(true_states, alpha=0.5, label='True State')
plt.plot(measurements, label='Measurements')
plt.plot(filtered_states, label='UKF Estimate')
plt.legend()
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.show()
